# msrest smoke tests

Exercises the primary `msrest` API surface: serialization, deserialization, configuration, authentication, and the service client.

In [ ]:
# Example 1: Serializer — serialize basic Python types into JSON-compatible primitives.
from msrest.serialization import Serializer

serializer = Serializer()

assert serializer.serialize_data("hello", "str") == "hello"
assert serializer.serialize_data(42, "int") == 42
assert serializer.serialize_data(3.14, "float") == 3.14
assert serializer.serialize_data(True, "bool") is True
assert serializer.serialize_data([1, 2, 3], "[int]") == [1, 2, 3]

print("Serializer primitive round-trips OK")

In [ ]:
# Example 2: Deserializer — parse ISO-8601 datetime + duration strings into Python objects.
import datetime
import isodate
from msrest.serialization import Deserializer

deserializer = Deserializer()

dt = deserializer.deserialize_iso("2024-01-15T10:30:00Z")
assert isinstance(dt, datetime.datetime)
assert dt.year == 2024 and dt.month == 1 and dt.day == 15

duration = deserializer.deserialize_duration("PT1H30M")
assert isinstance(duration, (datetime.timedelta, isodate.Duration))

print(f"Parsed datetime: {dt}")
print(f"Parsed duration: {duration}")

In [ ]:
# Example 4: Authentication — BasicTokenAuthentication signs a prepared request.
import requests
from msrest.authentication import BasicTokenAuthentication

token = {"access_token": "fake-token-abc123", "token_type": "Bearer"}
auth = BasicTokenAuthentication(token)

session = requests.Session()
signed = auth.signed_session(session)

req = requests.Request("GET", "https://example.com/api")
prepared = signed.prepare_request(req)

assert prepared.headers["Authorization"] == "Bearer fake-token-abc123"
print(f"Authorization header: {prepared.headers['Authorization']}")

In [ ]:
# Example 5: Model — define + serialize/deserialize a custom Model subclass.
from msrest.serialization import Model, Serializer, Deserializer

class Widget(Model):
    _attribute_map = {
        "name": {"key": "name", "type": "str"},
        "count": {"key": "count", "type": "int"},
    }

    def __init__(self, *, name=None, count=None, **kwargs):
        super().__init__(**kwargs)
        self.name = name
        self.count = count

client_models = {"Widget": Widget}
w = Widget(name="gear", count=7)
payload = Serializer(client_models).body(w, "Widget")
assert payload == {"name": "gear", "count": 7}

restored = Deserializer(client_models)("Widget", payload)
assert restored.name == "gear" and restored.count == 7
print(f"Round-tripped Widget: name={restored.name}, count={restored.count}")